# FactGenius Colab: chạy bằng folder CSV + Hugging Face/LLM

Notebook này là bản tự-contained để chạy trên Google Colab. Bạn không cần clone GitHub repo và không cần mang theo các file `.py` của dự án.

Bạn chỉ cần đặt folder dữ liệu CSV lên Colab hoặc Google Drive, ví dụ:

```text
/content/llm_v1_singleStage/train.csv
/content/llm_v1_singleStage/val.csv
/content/llm_v1_singleStage/test.csv
```

hoặc:

```text
/content/drive/MyDrive/FactGenius/llm_v1_singleStage/train.csv
/content/drive/MyDrive/FactGenius/llm_v1_singleStage/val.csv
/content/drive/MyDrive/FactGenius/llm_v1_singleStage/test.csv
```

Notebook có 2 hướng test/phân loại:

1. Load model đã train từ local folder hoặc Hugging Face rồi chạy trên `test.csv`.
2. Gọi LLM qua API key để chạy fact-check trên `test.csv`.

Nếu muốn, bạn cũng có thể fine-tune một model BERT/RoBERTa mới ngay trong notebook rồi lưu model vào `/content` hoặc Google Drive.

## 1. Kiểm tra runtime

Nếu train BERT/RoBERTa, hãy bật GPU: `Runtime > Change runtime type > T4 GPU` hoặc GPU tốt hơn. Nếu chỉ load model rồi test sample nhỏ, CPU vẫn chạy được nhưng chậm hơn.

In [ ]:
import sys
import platform

print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch chưa sẵn sàng:', repr(exc))

## 2. Cài thư viện

Các thư viện dưới đây đủ để đọc CSV, fine-tune/load model Hugging Face, evaluate trên test set, và gọi LLM qua OpenAI-compatible API.

In [ ]:
%pip -q install -U pandas numpy scikit-learn matplotlib tqdm transformers datasets accelerate openai

In [ ]:
import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 2024
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

try:
    import torch
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
except Exception:
    pass

print('Import xong.')

## 3. Chọn nơi đặt dữ liệu và nơi lưu model

Bạn chọn rõ dữ liệu nằm ở `/content` hay Google Drive. Notebook sẽ không clone repo.

- Nếu chọn `content`: upload folder CSV vào `/content`, ví dụ `/content/llm_v1_singleStage`.
- Nếu chọn `drive`: mount Drive và đặt folder CSV trong Drive, ví dụ `/content/drive/MyDrive/FactGenius/llm_v1_singleStage`.

Tương tự, model sau khi train có thể lưu ở `/content` hoặc Drive.

In [ ]:
# Chọn nơi chứa dữ liệu CSV.
DATA_STORAGE = 'content'  # @param ['content', 'drive']

# Chọn nơi lưu output/model nếu train mới.
OUTPUT_STORAGE = 'content'  # @param ['content', 'drive']

# Tên folder dữ liệu. Folder này phải chứa train.csv, val.csv, test.csv.
DATA_FOLDER_NAME = 'llm_v1_singleStage'  # @param ['llm_v1', 'llm_v1_singleStage'] {allow-input: true}

# Root nếu dùng /content.
CONTENT_DATA_ROOT = '/content'  # @param {type:'string'}
CONTENT_OUTPUT_ROOT = '/content/factgenius_outputs'  # @param {type:'string'}

# Root nếu dùng Google Drive.
DRIVE_DATA_ROOT = '/content/drive/MyDrive/FactGenius'  # @param {type:'string'}
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/FactGenius/outputs'  # @param {type:'string'}

if DATA_STORAGE == 'drive' or OUTPUT_STORAGE == 'drive':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ModuleNotFoundError:
        print('Không chạy trong Colab, bỏ qua mount Drive.')

data_root = Path(DRIVE_DATA_ROOT if DATA_STORAGE == 'drive' else CONTENT_DATA_ROOT)
output_root = Path(DRIVE_OUTPUT_ROOT if OUTPUT_STORAGE == 'drive' else CONTENT_OUTPUT_ROOT)

DATA_DIR = data_root / DATA_FOLDER_NAME
OUTPUT_DIR = output_root / DATA_FOLDER_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR   =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

### 3.1. Tùy chọn: upload file zip chứa folder CSV

Nếu bạn chưa đặt folder CSV vào `/content`, cách tiện nhất là zip folder `llm_v1_singleStage` hoặc `llm_v1`, upload lên Colab, rồi giải nén bằng cell này.

Ví dụ zip có cấu trúc:

```text
llm_v1_singleStage/train.csv
llm_v1_singleStage/val.csv
llm_v1_singleStage/test.csv
```

Nếu bạn đã có sẵn folder CSV đúng vị trí, bỏ qua cell này.

In [ ]:
UPLOAD_AND_UNZIP_DATA = False  # @param {type:'boolean'}

if UPLOAD_AND_UNZIP_DATA:
    try:
        from google.colab import files
        uploaded = files.upload()
        for filename in uploaded.keys():
            archive_path = Path('/content') / filename
            print('Giải nén:', archive_path)
            shutil.unpack_archive(str(archive_path), str(data_root))
        print('Giải nén xong vào:', data_root)
    except ModuleNotFoundError:
        print('Cell upload chỉ chạy trong Google Colab.')
else:
    print('Bỏ qua upload/unzip.')

## 4. Load và kiểm tra CSV

Mỗi file CSV cần có ít nhất 2 cột:

- `Sentence`: text đầu vào, thường có format `Claim: ... Evidence: ...`
- `Label`: nhãn `True/False` hoặc `1/0`

In [ ]:
train_path = DATA_DIR / 'train.csv'
val_path = DATA_DIR / 'val.csv'
test_path = DATA_DIR / 'test.csv'

for path in [train_path, val_path, test_path]:
    if not path.exists():
        raise FileNotFoundError(f'Không tìm thấy {path}. Hãy kiểm tra DATA_STORAGE, DATA_ROOT và DATA_FOLDER_NAME.')

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

required_columns = {'Sentence', 'Label'}
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f'{name}.csv thiếu cột: {missing}')

def normalize_label(value):
    if isinstance(value, (bool, np.bool_)):
        return int(value)
    text = str(value).strip().lower()
    if text in {'true', '1', 'yes'}:
        return 1
    if text in {'false', '0', 'no'}:
        return 0
    raise ValueError(f'Label không hợp lệ: {value!r}')

for df in [train_df, val_df, test_df]:
    df['labels'] = df['Label'].map(normalize_label).astype(int)

summary = pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'rows': [len(train_df), len(val_df), len(test_df)],
    'true': [int(train_df['labels'].sum()), int(val_df['labels'].sum()), int(test_df['labels'].sum())],
})
summary['false'] = summary['rows'] - summary['true']
display(summary)
display(train_df.head(3))

print('Ví dụ Sentence:')
print(train_df.iloc[0]['Sentence'])

## 5. Cấu hình model Hugging Face

Có 2 cách dùng:

- `TRAIN_NEW_MODEL=True`: fine-tune model nền như `bert-base-uncased`, sau đó lưu vào `TRAINED_MODEL_DIR`.
- `TRAIN_NEW_MODEL=False`: bỏ qua train, chỉ load model đã train từ local folder hoặc Hugging Face Hub để test.

Một số model đã được README gốc nhắc tới:

- `SushantGautam/KG-LLM-bert-base`
- `SushantGautam/KG-LLM-roberta-base`
- `SushantGautam/KG-LLM-roberta-base-single_stage`
- `SushantGautam/KG-LLM-roberta-base-claim_only`

In [ ]:
# Nếu True, notebook sẽ fine-tune model mới rồi lưu lại.
TRAIN_NEW_MODEL = False  # @param {type:'boolean'}

# Model nền khi train mới.
BASE_MODEL_NAME = 'bert-base-uncased'  # @param ['bert-base-uncased', 'roberta-base', 'distilbert-base-uncased'] {allow-input: true}

# Nơi lưu model sau khi train mới.
TRAINED_MODEL_DIR = str(OUTPUT_DIR / 'trained_hf_model')
print('TRAINED_MODEL_DIR =', TRAINED_MODEL_DIR)

# Nguồn model dùng để test: local folder hoặc Hugging Face Hub.
TEST_MODEL_SOURCE = 'huggingface'  # @param ['local', 'huggingface']

# Nếu TEST_MODEL_SOURCE='local', trỏ tới folder model đã save_pretrained.
LOCAL_TEST_MODEL_DIR = TRAINED_MODEL_DIR  # @param {type:'string'}

# Nếu TEST_MODEL_SOURCE='huggingface', điền model id trên Hugging Face.
HF_TEST_MODEL_ID = 'SushantGautam/KG-LLM-roberta-base-single_stage'  # @param {type:'string'}

# Chạy nhanh bằng sample nhỏ hay chạy full.
QUICK_RUN = True  # @param {type:'boolean'}
TRAIN_SAMPLE_SIZE = 4000  # @param {type:'integer'}
VAL_SAMPLE_SIZE = 1000  # @param {type:'integer'}
TEST_SAMPLE_SIZE = 1000  # @param {type:'integer'}

MAX_LENGTH = 256  # @param {type:'integer'}
EPOCHS = 1  # @param {type:'integer'}
BATCH_SIZE = 16  # @param {type:'integer'}
LEARNING_RATE = 2e-5  # @param {type:'number'}

In [ ]:
def sample_frame(df, n):
    if not QUICK_RUN or n <= 0 or n >= len(df):
        return df.copy().reset_index(drop=True)
    return df.sample(n=n, random_state=RANDOM_SEED).reset_index(drop=True)

train_run_df = sample_frame(train_df, TRAIN_SAMPLE_SIZE)
val_run_df = sample_frame(val_df, VAL_SAMPLE_SIZE)
test_run_df = sample_frame(test_df, TEST_SAMPLE_SIZE)

print('Train rows:', len(train_run_df))
print('Val rows:', len(val_run_df))
print('Test rows:', len(test_run_df))

## 6. Tùy chọn: fine-tune BERT/RoBERTa và lưu model

Nếu `TRAIN_NEW_MODEL=False`, cell này sẽ bỏ qua. Nếu bật train, model sẽ được lưu vào `TRAINED_MODEL_DIR` để cell test bên dưới load lại.

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import inspect
import torch

def build_dataset_dict(train_data, val_data, test_data):
    return DatasetDict({
        'train': Dataset.from_pandas(train_data[['Sentence', 'labels']], preserve_index=False),
        'validation': Dataset.from_pandas(val_data[['Sentence', 'labels']], preserve_index=False),
        'test': Dataset.from_pandas(test_data[['Sentence', 'labels']], preserve_index=False),
    })

def tokenize_dataset(dataset_dict, tokenizer, max_length):
    def tokenize_batch(batch):
        return tokenizer(batch['Sentence'], truncation=True, max_length=max_length)
    return dataset_dict.map(tokenize_batch, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

if TRAIN_NEW_MODEL:
    train_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    dataset_dict = build_dataset_dict(train_run_df, val_run_df, test_run_df)
    tokenized_datasets = tokenize_dataset(dataset_dict, train_tokenizer, MAX_LENGTH)
    data_collator = DataCollatorWithPadding(tokenizer=train_tokenizer)
    train_model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=2)

    training_kwargs = dict(
        output_dir=str(OUTPUT_DIR / 'trainer_runs'),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        logging_steps=50,
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        report_to='none',
        fp16=torch.cuda.is_available(),
    )

    training_params = inspect.signature(TrainingArguments.__init__).parameters
    if 'eval_strategy' in training_params:
        training_kwargs['eval_strategy'] = 'epoch'
    else:
        training_kwargs['evaluation_strategy'] = 'epoch'

    training_args = TrainingArguments(**training_kwargs)
    trainer = Trainer(
        model=train_model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['validation'],
        tokenizer=train_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()
    trainer.save_model(TRAINED_MODEL_DIR)
    train_tokenizer.save_pretrained(TRAINED_MODEL_DIR)
    print('Đã lưu model đã train tại:', TRAINED_MODEL_DIR)
else:
    print('Bỏ qua fine-tuning vì TRAIN_NEW_MODEL=False.')

## 7. Test trên `test.csv` bằng model đã train/load từ Hugging Face

Cell này là phần bạn cần khi đã có model. Nó sẽ load model từ:

- `LOCAL_TEST_MODEL_DIR` nếu `TEST_MODEL_SOURCE='local'`
- `HF_TEST_MODEL_ID` nếu `TEST_MODEL_SOURCE='huggingface'`

Sau đó chạy prediction trên `test.csv` và in classification report.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

RUN_HF_TEST = True  # @param {type:'boolean'}

if RUN_HF_TEST:
    model_id_or_path = LOCAL_TEST_MODEL_DIR if TEST_MODEL_SOURCE == 'local' else HF_TEST_MODEL_ID
    print('Loading model:', model_id_or_path)

    test_tokenizer = AutoTokenizer.from_pretrained(model_id_or_path)
    test_model = AutoModelForSequenceClassification.from_pretrained(model_id_or_path)

    hf_test_dataset = Dataset.from_pandas(test_run_df[['Sentence', 'labels']], preserve_index=False)

    def tokenize_test_batch(batch):
        return test_tokenizer(batch['Sentence'], truncation=True, max_length=MAX_LENGTH)

    hf_test_dataset = hf_test_dataset.map(tokenize_test_batch, batched=True)
    hf_data_collator = DataCollatorWithPadding(tokenizer=test_tokenizer)

    test_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / 'hf_test_tmp'),
        per_device_eval_batch_size=BATCH_SIZE,
        report_to='none',
    )

    test_trainer = Trainer(
        model=test_model,
        args=test_args,
        tokenizer=test_tokenizer,
        data_collator=hf_data_collator,
        compute_metrics=compute_metrics,
    )

    test_output = test_trainer.predict(hf_test_dataset)
    y_true = test_output.label_ids
    y_pred = np.argmax(test_output.predictions, axis=1)

    print('Metrics:', test_output.metrics)
    print('\nClassification report:')
    print(classification_report(y_true, y_pred, target_names=['False', 'True'], zero_division=0))
    print('Confusion matrix:')
    print(confusion_matrix(y_true, y_pred))

    hf_results_df = test_run_df.copy()
    hf_results_df['predicted_label'] = y_pred
    hf_results_df['gold_text'] = hf_results_df['labels'].map({0: 'False', 1: 'True'})
    hf_results_df['predicted_text'] = hf_results_df['predicted_label'].map({0: 'False', 1: 'True'})

    results_path = OUTPUT_DIR / f'hf_test_results_{DATA_FOLDER_NAME}.csv'
    hf_results_df.to_csv(results_path, index=False)
    print('Đã lưu kết quả test:', results_path)
    display(hf_results_df[['Sentence', 'gold_text', 'predicted_text']].head(10))
else:
    print('Bỏ qua Hugging Face test.')

### 7.1. Predict một vài câu bằng model Hugging Face

Dùng cell này để thử nhanh model đã load ở phần test. Nếu chưa chạy cell test ở trên, cell này sẽ tự load model lại.

In [ ]:
RUN_SINGLE_HF_INFERENCE = True  # @param {type:'boolean'}

def predict_with_hf_model(sentences, model=None, tokenizer=None):
    if model is None or tokenizer is None:
        model_id_or_path = LOCAL_TEST_MODEL_DIR if TEST_MODEL_SOURCE == 'local' else HF_TEST_MODEL_ID
        tokenizer = AutoTokenizer.from_pretrained(model_id_or_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_id_or_path)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    inputs = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        preds = probs.argmax(axis=1)

    return pd.DataFrame({
        'Sentence': sentences,
        'predicted_label': preds,
        'predicted_text': ['True' if pred == 1 else 'False' for pred in preds],
        'prob_false': probs[:, 0],
        'prob_true': probs[:, 1],
    })

if RUN_SINGLE_HF_INFERENCE:
    sample_sentences = test_df['Sentence'].head(3).tolist()
    display(predict_with_hf_model(sample_sentences, globals().get('test_model'), globals().get('test_tokenizer')))
else:
    print('Bỏ qua single inference.')

## 8. Test trên `test.csv` bằng LLM API

API key để trống để bạn tự cấu hình sau. Nếu dùng OpenAI, giữ `LLM_BASE_URL='https://api.openai.com/v1'`. Nếu dùng server riêng tương thích OpenAI như vLLM, đổi base URL, ví dụ `http://host:8000/v1`.

In [ ]:
from openai import OpenAI
import time

RUN_LLM_TEST = False  # @param {type:'boolean'}

LLM_API_KEY = ''  # @param {type:'string'}
LLM_BASE_URL = 'https://api.openai.com/v1'  # @param {type:'string'}
LLM_MODEL = 'gpt-4o-mini'  # @param {type:'string'}

# Nên test ít dòng trước để kiểm soát chi phí.
LLM_TEST_SAMPLE_SIZE = 20  # @param {type:'integer'}

def split_claim_evidence(sentence):
    text = str(sentence)
    if ' Evidence: ' in text:
        claim_part, evidence_part = text.split(' Evidence: ', 1)
        claim = claim_part.replace('Claim: ', '').strip()
        evidence = evidence_part.strip()
    else:
        claim = text.replace('Claim: ', '').strip()
        evidence = ''
    return claim, evidence

def parse_true_false(answer_text):
    clean = str(answer_text).strip()
    if not clean:
        return None
    first_line = clean.splitlines()[0].strip().lower()
    if first_line.startswith('true'):
        return 1
    if first_line.startswith('false'):
        return 0
    if 'true' in first_line and 'false' not in first_line:
        return 1
    if 'false' in first_line:
        return 0
    return None

def build_llm_messages(sentence):
    claim, evidence = split_claim_evidence(sentence)
    evidence_text = evidence if evidence else '(No external evidence provided.)'
    return [
        {
            'role': 'system',
            'content': 'You are a careful fact-checker. Decide whether the claim is supported. The first line must be exactly True or False.'
        },
        {
            'role': 'user',
            'content': f'''Claim:\n{claim}\n\nEvidence:\n{evidence_text}\n\nReturn exactly two lines:\nTrue or False\nOne short explanation.'''
        },
    ]

def llm_predict_one(client, sentence):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=build_llm_messages(sentence),
        temperature=0,
    )
    answer = response.choices[0].message.content
    return answer, parse_true_false(answer)

print('LLM config đã sẵn sàng. Có API key:', bool(LLM_API_KEY.strip()))

In [ ]:
if not RUN_LLM_TEST:
    print('Bỏ qua LLM test vì RUN_LLM_TEST=False.')
elif not LLM_API_KEY.strip():
    print('Chưa có LLM_API_KEY. Hãy điền API key rồi chạy lại cell.')
else:
    client = OpenAI(api_key=LLM_API_KEY.strip(), base_url=LLM_BASE_URL.strip())
    llm_test_df = sample_frame(test_df, LLM_TEST_SAMPLE_SIZE)
    rows = []

    for index, row in llm_test_df.iterrows():
        answer, pred = llm_predict_one(client, row['Sentence'])
        rows.append({
            'row': index,
            'Sentence': row['Sentence'],
            'gold_label': int(row['labels']),
            'predicted_label': pred,
            'gold_text': 'True' if int(row['labels']) == 1 else 'False',
            'predicted_text': None if pred is None else ('True' if pred == 1 else 'False'),
            'llm_answer': answer,
        })
        print(index, 'gold=', int(row['labels']), 'pred=', pred)
        time.sleep(0.2)

    llm_results_df = pd.DataFrame(rows)
    valid_llm_df = llm_results_df.dropna(subset=['predicted_label']).copy()

    if len(valid_llm_df):
        y_true = valid_llm_df['gold_label'].astype(int).to_numpy()
        y_pred = valid_llm_df['predicted_label'].astype(int).to_numpy()
        print('\nClassification report:')
        print(classification_report(y_true, y_pred, target_names=['False', 'True'], zero_division=0))
        print('Confusion matrix:')
        print(confusion_matrix(y_true, y_pred))
    else:
        print('Không parse được dự đoán True/False nào từ LLM output.')

    llm_results_path = OUTPUT_DIR / f'llm_test_results_{DATA_FOLDER_NAME}.csv'
    llm_results_df.to_csv(llm_results_path, index=False)
    print('Đã lưu kết quả LLM:', llm_results_path)
    display(llm_results_df.head(10))

## 9. Ghi chú chạy thực tế

- Nếu chỉ muốn test model đã train: đặt `TRAIN_NEW_MODEL=False`, chọn `TEST_MODEL_SOURCE`, rồi chạy tới mục 7.
- Nếu muốn train mới và lưu vào Drive: đặt `OUTPUT_STORAGE='drive'`, `TRAIN_NEW_MODEL=True`, rồi chạy mục 6.
- Nếu muốn test full `test.csv`: đặt `QUICK_RUN=False` hoặc đặt `TEST_SAMPLE_SIZE` lớn hơn số dòng test.
- Nếu dùng model Hugging Face đúng với biến thể dữ liệu, hãy chọn model tương ứng. Ví dụ dữ liệu `llm_v1_singleStage` nên thử `SushantGautam/KG-LLM-roberta-base-single_stage`; dữ liệu `llm_v1` nên thử `SushantGautam/KG-LLM-roberta-base` hoặc `SushantGautam/KG-LLM-bert-base`.
- LLM API nên chạy sample nhỏ trước vì có chi phí và phụ thuộc rate limit.